# Water sim

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from enum import Enum
from typing import Iterator


# ============================================================
# Basic types
# ============================================================

Scalar = float


@dataclass(frozen=True)
class Vector2:
    x: float
    y: float

    def __add__(self, other: "Vector2") -> "Vector2":
        return Vector2(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Vector2") -> "Vector2":
        return Vector2(self.x - other.x, self.y - other.y)

    def __mul__(self, value: float) -> "Vector2":
        return Vector2(self.x * value, self.y * value)

    def __truediv__(self, value: float) -> "Vector2":
        return Vector2(self.x / value, self.y / value)


@dataclass(frozen=True)
class Vector3:
    x: float
    y: float
    z: float


@dataclass(frozen=True)
class Index2:
    x: int
    y: int


@dataclass(frozen=True)
class Index3:
    x: int
    y: int
    z: int


# ============================================================
# Cell type
# ============================================================

class CellType(Enum):
    OUTSIDE = 0
    FLUID = 1
    SOLID = 2


# ============================================================
# Geometry
# ============================================================

class Geometry2D:
    """
    Describes the physical domain.

    Geometry does NOT know anything about velocity,
    pressure, density, etc.
    """

    def contains(self, position: Vector2) -> bool:
        """
        Return True if the position belongs to the simulation domain.
        """
        raise NotImplementedError


class RectangleGeometry(Geometry2D):
    def __init__(
        self,
        min_position: Vector2,
        max_position: Vector2,
    ):
        self.min_position = min_position
        self.max_position = max_position

    def contains(self, position: Vector2) -> bool:
        return (
            self.min_position.x <= position.x <= self.max_position.x
            and
            self.min_position.y <= position.y <= self.max_position.y
        )


class FlaskGeometry(Geometry2D):
    """
    Very simple flask-shaped geometry.

    This is only an example. Later we can create arbitrary
    geometries from polygons, meshes, SDFs, etc.
    """

    def contains(self, position: Vector2) -> bool:
        x = position.x
        y = position.y

        # Neck
        if 0.45 <= y <= 0.80:
            return 0.42 <= x <= 0.58

        # Body
        if 0.10 <= y < 0.45:
            width = 0.15 + (y - 0.10) * 0.85
            return (0.5 - width) <= x <= (0.5 + width)

        return False


# ============================================================
# Uniform Cartesian Grid
# ============================================================

class UniformGrid2D:
    """
    Regular Cartesian grid.

    Cell-centered quantities:
        pressure
        density
        cell_type

    MAC / staggered quantities:
        u -> X velocity on vertical faces
        v -> Y velocity on horizontal faces
    """

    def __init__(
        self,
        width: int,
        height: int,
        cell_size: float,
        geometry: Geometry2D,
    ):
        if width <= 0:
            raise ValueError("width must be positive")

        if height <= 0:
            raise ValueError("height must be positive")

        if cell_size <= 0:
            raise ValueError("cell_size must be positive")

        self.width = width
        self.height = height
        self.cell_size = cell_size
        self.geometry = geometry

        # ----------------------------------------------------
        # Cell-centered data
        # ----------------------------------------------------

        number_of_cells = width * height

        self.pressure = [0.0] * number_of_cells
        self.density = [0.0] * number_of_cells

        self.cell_type = [
            CellType.OUTSIDE
        ] * number_of_cells

        # ----------------------------------------------------
        # MAC velocity fields
        #
        # u has one more X-face than cells:
        #
        #   (width + 1) * height
        #
        # v has one more Y-face than cells:
        #
        #   width * (height + 1)
        # ----------------------------------------------------

        self.u = [0.0] * ((width + 1) * height)
        self.v = [0.0] * (width * (height + 1))

        self._initialize_cell_types()

    # ========================================================
    # Index helpers
    # ========================================================

    def cell_index(self, index: Index2) -> int:
        return index.x + index.y * self.width

    def u_index(self, x: int, y: int) -> int:
        return x + y * (self.width + 1)

    def v_index(self, x: int, y: int) -> int:
        return x + y * self.width

    # ========================================================
    # Position helpers
    # ========================================================

    def cell_center(self, index: Index2) -> Vector2:
        return Vector2(
            (index.x + 0.5) * self.cell_size,
            (index.y + 0.5) * self.cell_size,
        )

    # ========================================================
    # Initialization
    # ========================================================

    def _initialize_cell_types(self) -> None:
        for y in range(self.height):
            for x in range(self.width):
                index = Index2(x, y)

                position = self.cell_center(index)

                if self.geometry.contains(position):
                    self.set_cell_type(
                        index,
                        CellType.FLUID,
                    )

    # ========================================================
    # Cell access
    # ========================================================

    def is_inside(self, index: Index2) -> bool:
        return (
            0 <= index.x < self.width
            and
            0 <= index.y < self.height
        )

    def get_cell_type(self, index: Index2) -> CellType:
        if not self.is_inside(index):
            return CellType.OUTSIDE

        return self.cell_type[self.cell_index(index)]

    def set_cell_type(
        self,
        index: Index2,
        cell_type: CellType,
    ) -> None:
        if not self.is_inside(index):
            raise IndexError("Cell index outside grid")

        self.cell_type[self.cell_index(index)] = cell_type

    # ========================================================
    # Neighbors
    # ========================================================

    def neighbors(self, index: Index2) -> Iterator[Index2]:
        directions = (
            (-1, 0),
            (1, 0),
            (0, -1),
            (0, 1),
        )

        for dx, dy in directions:
            neighbor = Index2(
                index.x + dx,
                index.y + dy,
            )

            if self.is_inside(neighbor):
                yield neighbor

    # ========================================================
    # Debug / information
    # ========================================================

    def print_cell_types(self) -> None:
        """
        Simple ASCII visualization of the geometry.
        """

        symbols = {
            CellType.OUTSIDE: " ",
            CellType.FLUID: ".",
            CellType.SOLID: "#",
        }

        for y in reversed(range(self.height)):
            row = ""

            for x in range(self.width):
                cell = self.get_cell_type(Index2(x, y))
                row += symbols[cell]

            print(row)


# ============================================================
# Fluid state
# ============================================================

@dataclass
class FluidState2D:
    """
    Container for the physical state of the fluid.

    The grid owns the actual arrays.
    """

    grid: UniformGrid2D

    @property
    def pressure(self):
        return self.grid.pressure

    @property
    def density(self):
        return self.grid.density

    @property
    def u(self):
        return self.grid.u

    @property
    def v(self):
        return self.grid.v


# ============================================================
# Example
# ============================================================

def main():
    cell_size = 0.02

    width = 50
    height = 50

    geometry = FlaskGeometry()

    grid = UniformGrid2D(
        width=width,
        height=height,
        cell_size=cell_size,
        geometry=geometry,
    )

    fluid = FluidState2D(grid)

    # Add some initial density at the bottom of the flask.
    for y in range(height):
        for x in range(width):
            index = Index2(x, y)

            if grid.get_cell_type(index) == CellType.FLUID:

                position = grid.cell_center(index)

                if position.y < 0.20:
                    fluid.density[
                        grid.cell_index(index)
                    ] = 1.0

    # Show geometry.
    grid.print_cell_types()

    print()
    print("Grid:")
    print(f"  cells: {width} x {height}")
    print(f"  cell size: {cell_size}")

    print()
    print("Arrays:")
    print(f"  pressure: {len(grid.pressure)}")
    print(f"  density:  {len(grid.density)}")
    print(f"  u:        {len(grid.u)}")
    print(f"  v:        {len(grid.v)}")


if __name__ == "__main__":
    main()